<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
print(nltk.data.find('tokenizers/punkt'))

[nltk_data] Downloading package punkt to /home/jeffhe/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to /home/jeffhe/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jeffhe/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/jeffhe/nltk_data...


/home/jeffhe/nltk_data/tokenizers/punkt


In [2]:
import os
if os.path.basename(os.getcwd()) == "KE4MHQ":
    os.chdir("rome")
!ls


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution
import json
import time

from util.eval_greedy import eval_editing


import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)  # Python random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disable auto-optimization

set_seed(42)

 baselines		       globals.yml		    LICENSE
'both-Eval-[5]-[10]'	      'hop1-Eval-[10]'		    logs
'both-Eval-[5, 15]-[10, 20]'   Hop1-Eval-10		    notebooks
'both-Eval-[5]-[5]'	      'hop1-Eval-[5, 10, 15, 20]'   README.md
 CITATION.cff		       Hop1-Eval-5-10-15-20	    results
 data			      'hop2-Eval-[10]'		    rome
 dsets			       hop2-Eval-5-10-15-20	    scripts
 experiments		       hparams			    util


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [3]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
device = torch.device('cuda:0')
print(f"Using device: {device}")


Using device: cuda:0


In [ ]:
CUDA_VISIBLE_DEVICES=3 python3 -m experiments.evaluate \
    --alg_name=ROME-Multi \
    --model_name=EleutherAI/gpt-j-6B \
    --hparams_fname=EleutherAI_gpt-j-6B.json

In [5]:
!nvidia-smi

Fri Feb 28 21:56:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:31:00.0 Off |                  Off |
| 31%   55C    P2             73W /  300W |   41591MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory


In [5]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [4]:
ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
# layers_to_edit =[[5,15],[10,20]]
# layers_to_edit = [[5,10,15,20]]
layers_to_edit = [[10]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



In [5]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",

#  Pipeline for testing multiple insertions on MQuake

In [7]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[5,15],[10,20]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



edit_hop = "both" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 08:01:44
layers_to_edit:  [[5, 15], [10, 20]]
ds_file:  dsets/ds_classification/both_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 1
No model weights to restore: name 'orig_weights' is not defined

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5, 15], [10, 20]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


Cached context templates ['{}', 'Q: . {}', 'Q: . {}', 'The present invention relates. {}', 'The role of the. {}', '\n \n-. {}', 'A new report from. {}', 'Q: . {}', 'Q: . {}', '\n \n=. {}', 'Q: . {}', 'The present invention relates to a method for producing. {}', ' Ask HN: Is there any. {}', " Show HN: I'm looking. {}", 'Q: Is there a way to. {}', ' Show HN: The best way. {}', 'Q: How to make a list. {}', 'Q: How to use a function. {}', 'Q: How do you get the. {}', 'Q: Can I get the current. {}', 'Q: What is a good way. {}']
Computing left vector (u)...
Selected u projection object Fernando Santos
Retrieving inverse covariance statistics for EleutherAI_gpt-j-6B @ transformer.h.5.mlp.fc_out. The result will be cached to avoid repetitive computation.
Loading cached data/stats/EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.5.mlp.fc_out_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 3 | Sentence: Fernando Santos is a citizen of United | Token:  Santos
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 4.029 = 4.029 + 0.0 + 0.0 avg prob of [ United Kingdom] 0.018220696598291397
loss 1.795 = 1.721 + 0.049 + 0.025 avg prob of [ United Kingdom] 0.18069490790367126
loss 0.922 = 0.839 + 0.043 + 0.04 avg prob of [ United Kingdom] 0.4368959665298462
loss 0.436 = 0.335 + 0.049 + 0.052 avg prob of [ United Kingdom] 0.716685950756073
loss 0.25 = 0.144 + 0.044 + 0.063 avg prob of [ United Kingdom] 0.8668229579925537
loss 0.186 = 0.058 + 0.056 + 0.072 avg prob of [ United Kingdom] 0.9440300464630127
loss 0.153 = 0.025 + 0.049 + 0.079 avg prob of [ United Kingdom] 0.9754440784454346
loss 0.126 = 0.013 + 0.034 + 0.079 avg prob of [ United Kingdom] 0.9873984456062317
loss 0.112 = 0.008 + 0.026 + 0.079 avg prob of [ United Kingdom] 0.9923137426376343
loss 0.1

...

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 10 | Sentence: The name of the current head of state in United Kingdom is Emmerson Mnangag | Token:  Kingdom
Rewrite layer is 10
Tying optimization objective to 27
Recording initial value of v*
loss 1.995 = 1.995 + 0.0 + 0.0 avg prob of [ Emmerson Mnangagwa] 0.13722223043441772
loss 1.471 = 1.42 + 0.013 + 0.038 avg prob of [ Emmerson Mnangagwa] 0.2462625801563263
loss 1.163 = 1.094 + 0.011 + 0.058 avg prob of [ Emmerson Mnangagwa] 0.33869823813438416
loss 0.806 = 0.7 + 0.03 + 0.076 avg prob of [ Emmerson Mnangagwa] 0.49891212582588196
loss 0.606 = 0.481 + 0.033 + 0.091 avg prob of [ Emmerson Mnangagwa] 0.6194361448287964
loss 0.432 = 0.298 + 0.036 + 0.097 avg prob of [ Emmerson Mnangagwa] 0.7435059547424316
loss 0.244 = 0.111 + 0.035 + 0.097 avg prob of [ Emmerson Mnangagwa] 0.895510196685791
loss 0.148 = 0.016 + 0.035 + 0.097 avg prob of [ Emmerson Mnangagwa] 0.9843488335609436
loss 0.139 = 0.007 + 0

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 10 | Sentence: The name of the current head of state in United Kingdom is Emmerson Mnangag | Token:  Kingdom
Rewrite layer is 20
Tying optimization objective to 27
Recording initial value of v*
loss 1.995 = 1.995 + 0.0 + 0.0 avg prob of [ Emmerson Mnangagwa] 0.13722223043441772
loss 1.917 = 1.902 + 0.002 + 0.013 avg prob of [ Emmerson Mnangagwa] 0.15077441930770874
loss 1.551 = 1.523 + 0.007 + 0.021 avg prob of [ Emmerson Mnangagwa] 0.22076138854026794
loss 1.056 = 1.014 + 0.013 + 0.029 avg prob of [ Emmerson Mnangagwa] 0.36700499057769775
loss 0.586 = 0.531 + 0.02 + 0.036 avg prob of [ Emmerson Mnangagwa] 0.5944584608078003
loss 0.252 = 0.18 + 0.029 + 0.043 avg prob of [ Emmerson Mnangagwa] 0.8383417129516602
loss 0.129 = 0.044 + 0.035 + 0.05 avg prob of [ Emmerson Mnangagwa] 0.957556962966919
loss 0.111 = 0.023 + 0.031 + 0.056 avg prob of [ Emmerson Mnangagwa] 0.9769876003265381
loss 0.1 = 0.013 + 0

/home/jeffhe/anaconda3/envs/KE4MHQ_env/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Evaluation result saved to:  both-Eval-[5, 15]-[10, 20]/j-6B_id_1.json
Correct: 0/0


********************************************************************************************************************************************************************************************
Request 2, case_id: 5
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5, 15], [10, 20]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.

In [10]:
folder_dir = "Hop1-Eval-10"
correct = 0
total = len(os.listdir(folder_dir))
# iterate over the json files in the folder
for filename in os.listdir(folder_dir):
    if filename.endswith(".json"):
        with open(os.path.join(folder_dir, filename), "r") as f:
            data = json.load(f)
            if data[-1]["correct"]:
                correct += 1
print(f"Correct: {correct}/{total}")

Correct: 8/26


In [8]:
ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[5,10,15,20]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)


edit_hop = "hop1" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 09:19:12
layers_to_edit:  [[5, 10, 15, 20]]
ds_file:  dsets/ds_classification/hop1_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 2
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5, 10, 15, 20]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp

loss 0.133 = 0.004 + 0.036 + 0.094 avg prob of [ Benozzo Gozzoli] 0.9962979555130005
loss 0.132 = 0.003 + 0.035 + 0.094 avg prob of [ Benozzo Gozzoli] 0.9966839551925659
loss 0.131 = 0.003 + 0.034 + 0.094 avg prob of [ Benozzo Gozzoli] 0.9970154762268066
loss 0.13 = 0.003 + 0.034 + 0.094 avg prob of [ Benozzo Gozzoli] 0.9972928762435913
loss 0.129 = 0.002 + 0.033 + 0.094 avg prob of [ Benozzo Gozzoli] 0.9975215792655945
Delta norm: 85.54302215576172
Change in target norm: 21.38575553894043 to 88.62130737304688 => 67.23554992675781
Division Factor: 9.144623756408691
Right vector norm: 9.354460716247559
Right vector shape: torch.Size([4096])
Deltas successfully computed for 10
Executing ROME algorithm for the update: [Nick Bottom was created by] -> [ Benozzo Gozzoli]
Computing left vector (u)...
Selected u projection object Nick Bottom
Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 1 | Sentence: Nick Bottom was created by Benozzo Gozz | Token:  Bott

In [9]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[5],[10]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



edit_hop = "both" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 10:36:13
layers_to_edit:  [[5], [10]]
ds_file:  dsets/ds_classification/both_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 1
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5], [10]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_modu

In [10]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[5],[5]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



edit_hop = "both" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 11:20:58
layers_to_edit:  [[5], [5]]
ds_file:  dsets/ds_classification/both_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 1
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5], [5]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module

In [9]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[10]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



edit_hop = "hop2" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 15:33:20
layers_to_edit:  [[10]]
ds_file:  dsets/ds_classification/hop2_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 3
No model weights to restore: name 'orig_weights' is not defined

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[10]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='tr

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


Cached context templates ['{}', 'Q: . {}', 'Q: . {}', 'The present invention relates. {}', 'The role of the. {}', '\n \n-. {}', 'A new report from. {}', 'Q: . {}', 'Q: . {}', '\n \n=. {}', 'Q: . {}', 'The present invention relates to a method for producing. {}', ' Ask HN: Is there any. {}', " Show HN: I'm looking. {}", 'Q: Is there a way to. {}', ' Show HN: The best way. {}', 'Q: How to make a list. {}', 'Q: How to use a function. {}', 'Q: How do you get the. {}', 'Q: Can I get the current. {}', 'Q: What is a good way. {}']
Computing left vector (u)...
Selected u projection object London
Retrieving inverse covariance statistics for EleutherAI_gpt-j-6B @ transformer.h.10.mlp.fc_out. The result will be cached to avoid repetitive computation.
Loading cached data/stats/EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.10.mlp.fc_out_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 0 | Sentence: London is located in the continent of Euras | Token: London
Rewrite layer is 10
Tying optimization objective to 27
Recording initial value of v*
loss 2.493 = 2.493 + 0.0 + 0.0 avg prob of [ Eurasia] 0.08602311462163925
loss 1.858 = 1.83 + 0.01 + 0.018 avg prob of [ Eurasia] 0.16436073184013367
loss 1.043 = 0.996 + 0.019 + 0.028 avg prob of [ Eurasia] 0.382280170917511
loss 0.425 = 0.364 + 0.024 + 0.037 avg prob of [ Eurasia] 0.7107176780700684
loss 0.168 = 0.097 + 0.026 + 0.046 avg prob of [ Eurasia] 0.9292863607406616
loss 0.142 = 0.062 + 0.026 + 0.053 avg prob of [ Eurasia] 0.959811806678772
loss 0.145 = 0.054 + 0.03 + 0.06 avg prob of [ Eurasia] 0.9651665687561035
loss 0.148 = 0.05 + 0.031 + 0.067 avg prob of [ Eurasia] 0.9673736095428467
loss 0.145 = 0.049 + 0.028 + 0.068 avg prob of [ Eurasia] 0.9679926037788391
loss 0.143 = 0.048 + 0.027 + 0.068 avg prob of [ Eurasia] 0.96828389167

/home/jeffhe/anaconda3/envs/KE4MHQ_env/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Evaluation result saved to:  hop2-Eval-[10]/j-6B_id_3.json
Correct: 0/0


********************************************************************************************************************************************************************************************
Request 2, case_id: 7
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[10]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='

In [10]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit =[[10]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



edit_hop = "hop1" # choose one from ["hop1", "hop2", "both"]
# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

correct = 0
# for i in range(len(mhq_ds)):
for i in range(100):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

    print(f"Correct: {correct}/{i}")

print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

Start time:  2025-02-28 15:56:15
layers_to_edit:  [[10]]
ds_file:  dsets/ds_classification/hop1_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 2
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[10]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='tr

In [11]:

del model  # Deletes the model from memory
torch.cuda.empty_cache()  # Clears unused memory from the GPU
torch.cuda.ipc_collect()  # Helps reclaim unused memory
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Feb 28 16:19:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:31:00.0 Off |                  Off |
| 64%   72C    P2             33W /  300W |   25798MiB /  49140MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

: 